# 3. Reproducing biphenyl in 8CB, from your own transcription

This notebook reproduces a fit of **experimental** couplings: biphenyl dissolved in the liquid
crystal 8CB, measured by M. E. Di Pietro (PhD thesis, Université Paris-Sud / Università della
Calabria, 2013), **Table 3.1** (open copy: <http://hdl.handle.net/10955/1001>). Dipolar Studio
does not ship these values: transcribe the 12 couplings at **T_red = 0.930** yourself into a CSV
with the columns `spin_a, spin_b, D_obs_Hz, sigma_Hz`, one row per coupling, using the thesis
label → proton pair table below. Then set `COUPLINGS_CSV` to your file and run the notebook.

The model is the **symmetric four-site jump** (±φ, 180° ± φ, equal populations) on the built-in
biphenyl template. The reference values below are the fits that Dipolar Studio's independent
reference implementation gives for these data (`datasets/biphenyl-8CB-dipietro2013/expected/`,
frozen in M0); the thesis's own analysis (AP-DPD) agrees with them to 1–2 % in the order
parameters.

In [1]:
import csv
import os
from pathlib import Path

from dipolar_app.api import Studio

COUPLINGS_CSV = Path(os.environ.get("DIPOLAR_8CB_CSV", "my-8cb-T0.930.csv"))  # your transcription

# The thesis's labels and the protons of the template (ring A: H2-H6, ring B: H2'-H6').
THESIS_LABELS = {  # Table 3.1's label: the representative pair on the template
    "D12=D34": ("H2", "H3"),
    "D13=D13'": ("H2", "H2'"),
    "D11'=D33'": ("H2", "H6"),
    "D22'=D44'": ("H3", "H5"),
    "D14=D14'=D23=D23'": ("H2", "H3'"),
    "D24=D24'": ("H3", "H3'"),
    "D12'=D34'": ("H2", "H5"),
    "D16=D35": ("H2", "H4'"),
    "D15=D36": ("H2", "H4"),
    "D26=D45": ("H3", "H4'"),
    "D25=D46": ("H3", "H4"),
    "D56": ("H4", "H4'"),
}
if not COUPLINGS_CSV.exists():
    raise SystemExit(f"Transcribe Table 3.1 at T_red = 0.930 into {COUPLINGS_CSV} first (see above).")
rows = list(csv.DictReader(COUPLINGS_CSV.open(encoding="utf-8")))
print(len(rows), "couplings read from", COUPLINGS_CSV.name)
pairs = {tuple(sorted((r["spin_a"], r["spin_b"]))) for r in rows}
assert pairs == {tuple(sorted(p)) for p in THESIS_LABELS.values()}, "one row for each of Table 3.1's 12 labels"

12 couplings read from my-8cb-T0.930.csv


## Two fits

The thesis's σ are spectral-analysis errors, far smaller than the model's misfit, so two
configurations are fitted, as in the frozen reference:

- **unweighted**: every coupling with σ = 1 Hz, an ordinary least-squares fit (the one compared
  with the published values). With no error model behind σ, its χ², p-value and adequacy flag
  have no statistical meaning, and it reports no intervals: read its values only;
- **weighted**: the transcribed σ; the fit is then expected to be flagged **inadequate**
  (p < 0.001): the model does not explain the data within those σ.

In [2]:
REFERENCE = {  # Dipolar Studio's frozen reference fits at T_red = 0.930 (jump model)
    "unweighted": {"phi_deg": 32.932075, "Szz": 0.4307484, "Delta": 0.1852351},
    "weighted": {"phi_deg": 32.974574, "Szz": 0.4309707, "Delta": 0.1839539},
}
TOLERANCE = {"phi_deg": 1e-3, "Szz": 1e-5, "Delta": 1e-5}  # a transcription rounded as printed


def fit(sigma_of):
    table = "spin_a,spin_b,D_obs_Hz,sigma_Hz\n" + "".join(
        f"{r['spin_a']},{r['spin_b']},{r['D_obs_Hz']},{sigma_of(r)}\n" for r in rows
    )
    with Studio.local() as studio:
        studio.call("project.open_example", name="biphenyl-synthetic", expected_science_revision=0)
        proposal = studio.call("couplings.reconcile", csv=table)  # the template's labels match yours
        assert not proposal["unmatched_labels"], proposal["unmatched_labels"]
        studio.call("couplings.import", csv=table, confirmed_mapping=proposal["proposed_mapping"],
                    expected_science_revision=studio.science_revision)
        studio.call("model.set", kind="jump4", order="bisector-diagonal", frame="bisector",
                    phi_domain=[{"value": 0, "unit": "deg"}, {"value": 90, "unit": "deg"}],
                    expected_science_revision=studio.science_revision)
        return studio.fit()


results = {"unweighted": fit(lambda r: 1), "weighted": fit(lambda r: r["sigma_Hz"])}
for name, run in results.items():
    values = {**{t["name"]: t["value"] for t in run["theta"]}, **{o["name"]: o["value"] for o in run["order"]}}
    p = run["p_value"]
    p_text = f"p = {p:.1e}" if p > 0 else "p < 1e-300"  # chi2 in the hundred thousands: p underflows
    verdict = f"  adequate: {run['adequate']} ({p_text})" if name == "weighted" else ""
    print(f"{name:10}  phi = {values['phi_deg']:.4f} deg  S_zz = {values['Szz']:.5f}  Delta = {values['Delta']:.5f}"
          f"  RMS = {run['rms_residual']['value']:.2f} Hz{verdict}")
    for key, expected in REFERENCE[name].items():
        assert abs(values[key] - expected) <= TOLERANCE[key], (name, key, values[key], expected)
assert results["weighted"]["adequate"] is False
print("reproduced: both fits agree with the reference.")

unweighted  phi = 32.9321 deg  S_zz = 0.43075  Delta = 0.18524  RMS = 5.22 Hz
weighted    phi = 32.9746 deg  S_zz = 0.43097  Delta = 0.18395  RMS = 5.40 Hz  adequate: False (p < 1e-300)
reproduced: both fits agree with the reference.
